In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

1. Import the necessary modules

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

2. Accessing the dataset

In [23]:
data_path = "/kaggle/input/speech-emotion-recognition-using-ASED-dataset"

for root, dirs, files in os.walk(data_path):
    print(root, len(files))

3. Build Label Mapping

In [21]:
label_map = {}
labels = os.listdir(data_path)

for i, label in enumerate(labels):
    label_map[label] = i

print(label_map)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/speech-emotion-recognition-using-ASED-dataset'

4. feature extraction(LOG-MEL)

In [24]:
def extract_features(file_path, max_len=128, n_mels=128):
    audio, sr = librosa.load(file_path, sr=16000)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=n_mels
    )

    log_mel = librosa.power_to_db(mel)

    # padding / truncation
    if log_mel.shape[1] < max_len:
        pad = max_len - log_mel.shape[1]
        log_mel = np.pad(log_mel, ((0,0),(0,pad)))
    else:
        log_mel = log_mel[:, :max_len]

    return log_mel

5. Create dataset class

In [25]:
class SERDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        label = self.labels[idx]

        features = extract_features(file_path)

        features = torch.tensor(features, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(label, dtype=torch.long)

        return features, label

6. prepare data split 

In [26]:
files = []
labels = []

for label in os.listdir(data_path):
    folder = os.path.join(data_path, label)

    for f in os.listdir(folder):
        if f.endswith(".wav"):
            files.append(os.path.join(folder, f))
            labels.append(label_map[label])

X_train, X_val, y_train, y_val = train_test_split(
    files, labels, test_size=0.2, random_state=42
)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/speech-emotion-recognition-using-ASED-dataset'

7. Model (CNN + BiLSTM)

In [27]:
class CNN_BiLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm = nn.LSTM(
            input_size=64 * 32,
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)

        b, c, h, w = x.shape
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, w, c * h)

        x, _ = self.lstm(x)

        x = x[:, -1, :]

        return self.fc(x)

8. Training setup

In [28]:
train_dataset = SERDataset(X_train, y_train)
val_dataset = SERDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

NameError: name 'X_train' is not defined

9. Train loop

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_BiLSTM(num_classes=len(label_map)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(15):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        outputs = model(x)
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


NameError: name 'train_loader' is not defined

10. Validation

In [30]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

print("Validation Accuracy:", correct / total)

NameError: name 'val_loader' is not defined